# Telecom Customer Churn Data Cleaning & Logistic Regression Preparation

## Project Overview
This notebook documents the end-to-end process of cleaning a telecom customer dataset, preparing it for machine learning, training a logistic regression model, and evaluating its performance. The workflow preserves the original analysis while improving data quality and ensuring the dataset is suitable for predictive modeling.

## Import Libraries
Import the Python libraries required for data manipulation, preprocessing, modeling, and evaluation.

In [1]:
import pandas as pd
import numpy as np
import os
 
print(f"Using Pandas {pd.__version__} and NumPy {np.__version__}")
 
# find the raw file automatically instead of hardcoding a path,
# so this still runs even if I move the notebook around
target_raw_file = "TelecomChurn_Nigeria_Raw.csv"
found_raw_path = None
 
for root, dirs, files in os.walk('.', topdown=False):
    if target_raw_file in files:
        found_raw_path = os.path.join(root, target_raw_file)
        break
 
if not found_raw_path:
    for root, dirs, files in os.walk('..', topdown=False):
        if target_raw_file in files:
            found_raw_path = os.path.join(root, target_raw_file)
            break
 
if not found_raw_path:
    found_raw_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', target_raw_file))
 
print(f"Targeting raw dataset path: {found_raw_path}")
 
# reading everything in as strings for now so I don't lose things like
# the ₦ symbol or % signs before I've had a chance to clean them properly
df = pd.read_csv(found_raw_path, dtype=str)
df['Customer_Since_Raw'] = df['Customer_Since']  # keeping a backup of the original date strings
 
print(f"Success! Raw data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns.")

Using Pandas 2.3.3 and NumPy 2.3.5
Targeting raw dataset path: .\Telecom_Project\Data\TelecomChurn_Nigeria_Raw.csv
Success! Raw data loaded: 10,150 rows, 40 columns.


## Load Dataset
Load the dataset into the notebook and perform an initial check to confirm it was imported successfully.

In [2]:
# Looking at the first 5 rows 
df.head()


,Customer_ID,Full_Name,Gender,Age,State,Customer_Since,Marital_Status,Employment_Status,Monthly_Income,Plan_Type,...,Referral_Count,Promotion_Clicked,Email_Open_Rate,Contract_Type,Tenure_Months,Estimated_CLV,Risk_Score,Customer_Segment,Churn,Customer_Since_Raw
0,CUST-101643,Nozi Afolabi,Female,69,NASARAWA,3/2/2015,Single,Unemployed,"12,739",Premium,...,0,No,67,Two-Year,125,"NGN 210,969",30,Premium,No,3/2/2015
1,CUST-108819,Chidinma Agwu,Male,29,Bauchii,6/8/2021,single,Freelancer,"428,380",Premum,...,2,No,8,Monthly,144,"281,265",19,Corporate,No,6/8/2021
2,CUST-103718,Ruth Adekoya,MALE,36,Abuj,NaN,Divorced,Retired,"₦84,850",Premum,...,1,No,0.78,Annual,14,"NGN 25,558",44,Premium,No,NaN
3,CUST-102474,Bello Abubakar,female,73,Edo,9/25/2021,Single,Self-employed,"383,979",Standerd,...,0,No,23.90%,Annual,44,24222,31,At Risk,Yes,9/25/2021
4,CUST-105346,Victor Agwu,F,29,Imo State,12/3/2017,Married,Self-employed,NaN,premium,...,0,No,0.38,Two-Year,93,"182,955",17,Premium,No,12/3/2017


In [3]:
# Checking column shapes and data types
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10150 entries, 0 to 10149
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Customer_ID                  10150 non-null  object
 1   Full_Name                    10150 non-null  object
 2   Gender                       10026 non-null  object
 3   Age                          10150 non-null  object
 4   State                        10150 non-null  object
 5   Customer_Since               9747 non-null   object
 6   Marital_Status               9736 non-null   object
 7   Employment_Status            9830 non-null   object
 8   Monthly_Income               9064 non-null   object
 9   Plan_Type                    10150 non-null  object
 10  Internet_Type                10150 non-null  object
 11  Monthly_Data_Usage_GB        10150 non-null  object
 12  Monthly_Call_Minutes         10150 non-null  object
 13  SMS_Usage                    10

## Initial Data Inspection
Review the dataset structure, data types, and sample records to understand its current state before cleaning.

In [4]:
# The dataset has multiple text placeholders for empty fields. 
# Turning them all into actual missing values (NaN) so we can count them easily.
bad_blanks = ['N/A', 'NULL', 'null', 'none', 'None', 'nan', 'NaN', '', ' ']
df.replace(bad_blanks, np.nan, inplace=True)

print("Standardized all blank spaces successfully.")


Standardized all blank spaces successfully.


In [5]:
# Checking the final count of missing values per column
print("Missing fields per column (Top 15):")
print(df.isnull().sum().sort_values(ascending=False).head(15))


Missing fields per column (Top 15):
Monthly_Income           1086
Estimated_CLV             934
Outstanding_Balance       866
Monthly_Bill              861
Discount_Applied          500
Complaint_Category        494
Customer_Satisfaction     490
Marital_Status            414
Customer_Since_Raw        403
Customer_Since            403
Roaming_Usage             396
Email_Open_Rate           385
Employment_Status         320
Gender                    124
Risk_Score                  0
dtype: int64


## Data Cleaning
Standardize missing values, fix inconsistent formatting, correct invalid entries, and prepare the data for analysis while preserving the underlying information.

In [6]:
# Strip spaces and force all custom IDs into a strict CUST-XXXXXX format
df['Customer_ID'] = df['Customer_ID'].str.strip()

def clean_id_format(val):
    if pd.isna(val): 
        return np.nan
    # Isolate the digits by stripping different prefix variations
    just_digits = str(val).lower().replace('cust-', '').replace('cust', '').strip()
    return f"CUST-{just_digits}" if just_digits.isdigit() else val

df['Customer_ID'] = df['Customer_ID'].apply(clean_id_format)

# Drop duplicate accounts and check how many lines we drop
rows_before = len(df)
df = df.drop_duplicates(subset='Customer_ID', keep='first')
rows_after = len(df)

print(f"Dropped {rows_before - rows_after} duplicate customer accounts.")
print(f"Verified clean rows remaining: {rows_after:,}")


Dropped 150 duplicate customer accounts.
Verified clean rows remaining: 10,000


In [7]:
# Cleaning spaces and forcing standard Title Case formatting on all subscriber names
df['Full_Name'] = df['Full_Name'].str.strip().str.replace(r'/s+', ' ', regex=True).str.title()

print("Sample of cleaned names:")
for name in df['Full_Name'].head(5).tolist():
    print(f" - {name}")


Sample of cleaned names:
 - Nozi Afolabi
 - Chidinma Agwu
 - Ruth Adekoya
 - Bello Abubakar
 - Victor Agwu


In [8]:
# The first 5 rows
df.head()

,Customer_ID,Full_Name,Gender,Age,State,Customer_Since,Marital_Status,Employment_Status,Monthly_Income,Plan_Type,...,Referral_Count,Promotion_Clicked,Email_Open_Rate,Contract_Type,Tenure_Months,Estimated_CLV,Risk_Score,Customer_Segment,Churn,Customer_Since_Raw
0,CUST-101643,Nozi Afolabi,Female,69,NASARAWA,3/2/2015,Single,Unemployed,"12,739",Premium,...,0,No,67,Two-Year,125,"NGN 210,969",30,Premium,No,3/2/2015
1,CUST-108819,Chidinma Agwu,Male,29,Bauchii,6/8/2021,single,Freelancer,"428,380",Premum,...,2,No,8,Monthly,144,"281,265",19,Corporate,No,6/8/2021
2,CUST-103718,Ruth Adekoya,MALE,36,Abuj,NaN,Divorced,Retired,"₦84,850",Premum,...,1,No,0.78,Annual,14,"NGN 25,558",44,Premium,No,NaN
3,CUST-102474,Bello Abubakar,female,73,Edo,9/25/2021,Single,Self-employed,"383,979",Standerd,...,0,No,23.90%,Annual,44,24222,31,At Risk,Yes,9/25/2021
4,CUST-105346,Victor Agwu,F,29,Imo State,12/3/2017,Married,Self-employed,NaN,premium,...,0,No,0.38,Two-Year,93,"182,955",17,Premium,No,12/3/2017


In [9]:
# Standardizing demographic groupings to remove shorthand entries and typos
gender_clean = {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'}
marital_clean = {'maried': 'Married', 'divorsed': 'Divorced'}

df['Gender'] = df['Gender'].str.strip().str.lower().map(gender_clean).fillna('Unknown')

# For marital status, we patch the typos first, then convert the rest to Title Case
df['Marital_Status'] = df['Marital_Status'].str.strip().str.lower()
df['Marital_Status'] = df['Marital_Status'].replace(marital_clean).str.title().fillna('Unknown')

print("--- DEMOGRAPHIC COUNTS ---")
print(df['Gender'].value_counts())
print("/n", df['Marital_Status'].value_counts(), sep="")


--- DEMOGRAPHIC COUNTS ---
Gender
Male       4824
Female     4785
Unknown     391
Name: count, dtype: int64

Marital_Status
Single      2766
Married     2728
Divorced    2699
Widowed     1399
Unknown      408
Name: count, dtype: int64


In [10]:
# Forcing all states to lowercase first to eliminate any casing discrepancies
df['State'] = df['State'].str.strip().str.lower()

# Direct typo-mapping dictionary to merge manual entry errors into standard states
nigerian_state_fixes = {
    'bornu': 'Borno', 'kanno': 'Kano', 'enuguu': 'Enugu', 'anabra': 'Anambra',
    'river': 'Rivers', 'river state': 'Rivers', 'rivers state': 'Rivers',
    'oyoo': 'Oyo', 'ogunn': 'Ogun', 'lagoss': 'Lagos', 'lago': 'Lagos',
    'kduna': 'Kaduna', 'kadunna': 'Kaduna', 'cross rivers': 'Cross River',
    'crossriver': 'Cross River', 'cross river state': 'Cross River',
    'akwa-ibom': 'Akwa Ibom', 'akwaibom': 'Akwa Ibom', 'akwa ibom state': 'Akwa Ibom',
    'fct': 'Abuja', 'federal capital territory': 'Abuja', 'abuja fct': 'Abuja', 'abuj': 'Abuja'
}

# Apply the fixes and remove the word ' state' systematically if it exists
df['State'] = df['State'].replace(nigerian_state_fixes).str.replace(' state', '', case=False).str.title().fillna('Unknown')

print("Top 10 States after consolidation:")
print(df['State'].value_counts().head(10))
print(f"/nRemaining unparsed / Unknown states: {(df['State'] == 'Unknown').sum()}")


Top 10 States after consolidation:
State
Abuja        539
Taraba       290
Gombe        289
Adamawa      288
Borno        287
Ogun         286
Ondo         285
Kano         283
Akwa Ibom    282
Sokoto       273
Name: count, dtype: int64

Remaining unparsed / Unknown states: 0


In [11]:
# Mapping the dataset's generic tiers to actual retail packages used in Nigeria
retail_plan_mapping = {
    "basic": "Daily Plan", 
    "basik": "Daily Plan", 
    "daily/weekly bundle": "Daily Plan",
    "standard": "Weekly Plan", 
    "standerd": "Weekly Plan", 
    "prepaid monthly bundle": "Weekly Plan",
    "premium": "Monthly Plan", 
    "premum": "Monthly Plan", 
    "high value bundle": "Monthly Plan",
    "business": "Social / Bing Bundle", 
    "busines": "Social / Bing Bundle", 
    "sme shared plan": "Social / Bing Bundle",
    "enterprise": "SME Shared Data", 
    "corporate cug": "SME Shared Data",
    "unlimited": "Unlimited Router Plan", 
    "unlimted": "Unlimited Router Plan", 
    "unlimited broadband": "Unlimited Router Plan",
    "legacy": "Pay-As-You-Go (PAYG)", 
    "payg (pay-as-you-go)": "Pay-As-You-Go (PAYG)"
}

# Standardize the text case to lowercase first to ensure perfect dictionary matches
df['Plan_Type'] = df['Plan_Type'].str.strip().str.lower().map(retail_plan_mapping).fillna('Unknown')

print("--- LOCALIZED RETAILED PLANS ---")
print(df['Plan_Type'].value_counts())


--- LOCALIZED RETAILED PLANS ---
Plan_Type
Weekly Plan              2997
Daily Plan               2469
Monthly Plan             2004
Social / Bing Bundle     1000
Pay-As-You-Go (PAYG)      527
Unlimited Router Plan     503
SME Shared Data           500
Name: count, dtype: int64


In [12]:
# Convert Age values to numbers and drop impossible anomalies
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df.loc[(df['Age'] < 16) | (df['Age'] > 90), 'Age'] = np.nan

# Fill empty age slots with the column median
df['Age'] = df['Age'].fillna(df['Age'].median()).astype(int)

# Fix logic clash: Grandparents shouldn't be tagged as students or interns
df.loc[(df['Age'] > 30) & (df['Employment_Status'].str.strip().str.title().isin(['Student', 'Intern'])), 'Employment_Status'] = 'Self-Employed'

print(f"Cleaned Age Profile Range: {df['Age'].min()} - {df['Age'].max()} Years old")
print(f"Median Age verified: {df['Age'].median()}")


Cleaned Age Profile Range: 18 - 75 Years old
Median Age verified: 47.0


In [13]:
# Converting raw string records into true datetime objects using native mixed-format parsing
df['Customer_Since'] = pd.to_datetime(df['Customer_Since_Raw'], format='mixed', errors='coerce')

# Establishing my project reference timeline anchor
baseline_reference = pd.Timestamp('2025-06-01')
df['Tenure_Months'] = pd.to_numeric(df['Tenure_Months'], errors='coerce').fillna(1)

# Backfilling the 398 missing dates by projecting backward from known tenure months
missing_dates_mask = df['Customer_Since'].isnull()
days_offset = (df.loc[missing_dates_mask, 'Tenure_Months'] * 30.44).astype(int)
df.loc[missing_dates_mask, 'Customer_Since'] = baseline_reference - pd.to_timedelta(days_offset, unit='D')

# Recalculate true, definitive tenure months based on the newly standardized timestamps
df['Tenure_Months'] = ((baseline_reference - df['Customer_Since']).dt.days / 30.44).round(0).clip(lower=1)

print(f"Calculated Tenure Range: {df['Tenure_Months'].min()} to {df['Tenure_Months'].max()} months")
print(f"Remaining Missing Dates: {df['Customer_Since'].isnull().sum()}")


Calculated Tenure Range: 1.0 to 194.0 months
Remaining Missing Dates: 0


In [14]:
# STRIPPING CURRENCY UTILITIES AND DECIMALS
# Removing text formatting characters to convert columns into raw numerical values.

def strip_currency_markers(text_val):
    if pd.isna(text_val): 
        return np.nan
    # Remove text metrics markers and symbols safely
    cleaned_string = str(text_val).replace('₦', '').replace('NGN', '').replace(',', '').strip()
    try:
        numeric_output = float(cleaned_string)
        return np.nan if numeric_output < 0 else numeric_output
    except:
        return np.nan

# Define the financial metrics columns array
financial_metrics = ['Monthly_Income', 'Monthly_Bill', 'Outstanding_Balance', 'Estimated_CLV']

for col in financial_metrics:
    df[col] = df[col].apply(strip_currency_markers)
    df[col] = df[col].fillna(df[col].median())

print("Financial metrics converted to numbers successfully:")
for col in financial_metrics:
    print(f" - {col}: Median = ₦{df[col].median():,.2f}")


Financial metrics converted to numbers successfully:
 - Monthly_Income: Median = ₦119,347.50
 - Monthly_Bill: Median = ₦5,228.00
 - Outstanding_Balance: Median = ₦3,757.00
 - Estimated_CLV: Median = ₦41,021.00


In [15]:
# Check exactly what unique text strings are sitting inside the Contract column right now
print(df['Contract_Type'].unique())


['Two-Year' 'Monthly' 'Annual' 'Legacy']


In [16]:
# Inspecting the raw text and row count inside Contract_Type
print("Unique values in column:", df['Contract_Type'].unique())
print("First 5 rows of data:")
print(df['Contract_Type'].head(5))


Unique values in column: ['Two-Year' 'Monthly' 'Annual' 'Legacy']
First 5 rows of data:
0    Two-Year
1     Monthly
2      Annual
3      Annual
4    Two-Year
Name: Contract_Type, dtype: object


In [17]:
# rebuilding contract type and customer segment from the plan type,
# since the original values were a mess of AI-generated labels

print("Running contract re-mapping and applying line boundaries...")

np.random.seed(42)        

# Force everything to clean lowercase string text to clear spacing errors
df['Plan_Type'] = df['Plan_Type'].astype(str).str.strip().str.lower()

# LOGICAL RECONSTRUCTION OF CONTRACT TYPES AND SEGMENTS
is_corporate = df['Plan_Type'].str.contains('sme|shared|cug', na=False)
is_high_value = df['Plan_Type'].str.contains('monthly|unlimited|router|broadband', na=False)

# defaulting everyone to prepaid, then upgrading based on plan type
df['Contract_Type'] = 'Prepaid Ad-hoc'  
df.loc[is_corporate, 'Contract_Type'] = 'Corporate Postpaid Lease'
df.loc[is_high_value, 'Contract_Type'] = 'Individual Postpaid'

# same idea for customer segment
df['Customer_Segment'] = 'Prepaid Mass Market'  
df.loc[is_corporate, 'Customer_Segment'] = 'Corporate - SME Account'
df.loc[df['Plan_Type'].str.contains('corporate', na=False), 'Customer_Segment'] = 'Corporate - Key Enterprise Account'
df.loc[df['Plan_Type'] == 'monthly plan', 'Customer_Segment'] = 'HVC (High Value Customers)'
df.loc[df['Plan_Type'] == 'social / bing bundle', 'Customer_Segment'] = 'Multi-SIM Price Sensitive'

# renaming internet types to labels that actually make sense for a Nigerian telecom
df['Internet_Type'] = df['Internet_Type'].astype(str).str.strip().replace({"Dsl": "2G/3G Mobile Data", "DSL": "2G/3G Mobile Data", "Satellite": "Fixed Wireless (MiFi)", "No Internet": "Voice Only (2G)", "Fiber": "FTTH (Fiber to the Home)", "4G": "4G LTE", "5G": "5G Mobile"}).fillna('Unknown')

# making sure income and bill are numeric with sensible fallback values
df['Monthly_Income'] = pd.to_numeric(df['Monthly_Income'], errors='coerce').fillna(119347)
df['Monthly_Bill'] = pd.to_numeric(df['Monthly_Bill'], errors='coerce').fillna(5228)

# Infrastructure Checks
df.loc[df['Internet_Type'] == 'Voice Only (2G)', ['Monthly_Data_Usage_GB', 'Streaming_Hours']] = 0.0

# Fixing modern network tenure limits using row counts
mask_5g = (df['Internet_Type'] == '5G Mobile') & (df['Tenure_Months'] > 48)
rows_5g = len(df[mask_5g])
if rows_5g > 0:
    df.loc[mask_5g, 'Tenure_Months'] = np.random.randint(12, 48, size=rows_5g)

mask_4g = (df['Internet_Type'] == '4G LTE') & (df['Tenure_Months'] > 110)
rows_4g = len(df[mask_4g])
if rows_4g > 0:
    df.loc[mask_4g, 'Tenure_Months'] = np.random.randint(24, 110, size=rows_4g)

# Fixing Income ranges to clear out extreme outliers
mask_low_income = df['Monthly_Income'] < 35000
rows_low_income = len(df[mask_low_income])
if rows_low_income > 0:
    df.loc[mask_low_income, 'Monthly_Income'] = np.random.randint(35000, 75000, size=rows_low_income)

is_consumer = ~df['Customer_Segment'].isin(['Corporate - Key Enterprise Account', 'Corporate - SME Account'])
mask_high_income = is_consumer & (df['Monthly_Income'] > 1500000)
rows_high_income = len(df[mask_high_income])
if rows_high_income > 0:
    df.loc[mask_high_income, 'Monthly_Income'] = np.random.randint(500000, 1200000, size=rows_high_income)

# 6. Restoring realistic debt balances to postpaid users while keeping prepaid at 0
is_postpaid = df['Contract_Type'].str.lower().str.contains('postpaid', na=False)
is_prepaid = ~is_postpaid
postpaid_total = len(df[is_postpaid])

np.random.seed(42)
df.loc[is_postpaid, 'Outstanding_Balance'] = df.loc[is_postpaid, 'Monthly_Bill'] * np.random.uniform(0.1, 2.5, size=postpaid_total)
df.loc[is_prepaid, 'Outstanding_Balance'] = 0.0
df['Outstanding_Balance'] = df['Outstanding_Balance'].round(2)

# Clean corporate title case and correcting abbreviations
df['Plan_Type'] = df['Plan_Type'].str.title()
df['Plan_Type'] = df['Plan_Type'].replace({
    'Pay-As-You-Go (Payg)': 'PAYG',
    'Sme Shared Data': 'SME Shared Data' 
})

print("/n  DISTRIBUTION CHECK")
print(f"Clean Unique Plans: {df['Plan_Type'].unique().tolist()}")
print("/nContract Balance Breakdown:")
print(df['Contract_Type'].value_counts())


Running contract re-mapping and applying line boundaries...

  DISTRIBUTION CHECK
Clean Unique Plans: ['Monthly Plan', 'Weekly Plan', 'PAYG', 'Daily Plan', 'Social / Bing Bundle', 'SME Shared Data', 'Unlimited Router Plan']

Contract Balance Breakdown:
Contract_Type
Prepaid Ad-hoc              6993
Individual Postpaid         2507
Corporate Postpaid Lease     500
Name: count, dtype: int64


In [18]:
# Converting columns from text strings to actual numbers.
# Then scaling down the inflated AI metrics to look realistic.

print("Standardizing behavioral metrics and scaling complaint columns...")

# List of columns that need to be numbers
numeric_behavioral = [
    'Monthly_Call_Minutes', 'SMS_Usage', 'Streaming_Hours', 
    'Late_Payment_Count', 'Support_Tickets', 'Customer_Satisfaction', 
    'NPS_Score', 'Mobile_App_Login_Last30Days', 'Website_Visits', 'Referral_Count'
]

# Loop through and explicitly force each column to numbers FIRST, then handle blanks
for col in numeric_behavioral:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Now that it is safely a numeric column, we can calculate a proper median
    df[col] = df[col].fillna(df[col].median())

# Clean email open rate formatting percentages
def fix_email_rate(val):
    if pd.isna(val): 
        return np.nan
    clean_str = str(val).strip()
    if clean_str.endswith('%'):
        try: return float(clean_str.replace('%', '')) / 100
        except: return np.nan
    try:
        num = float(clean_str)
        return num / 100 if num > 1 else num
    except:
        return np.nan

df['Email_Open_Rate'] = df['Email_Open_Rate'].apply(fix_email_rate)
df['Email_Open_Rate'] = df['Email_Open_Rate'].fillna(df['Email_Open_Rate'].median()).clip(0, 1)

# Clean subscription add-on text separators
def clean_subscription_strings(val):
    if pd.isna(val): 
        return 'None'
    clean_str = str(val).strip()
    if clean_str.lower() in ['yes', 'no']: 
        return 'Yes' if clean_str.lower() == 'yes' else 'None'
    import re
    clean_str = re.sub(r'[;|]', ',', clean_str)
    items = [item.strip() for item in clean_str.split(',') if item.strip()]
    return ', '.join(sorted(set(items)))

df['Add_On_Subscription'] = df['Add_On_Subscription'].apply(clean_subscription_strings)

# These two columns were way higher than realistic telecom numbers, so scaling them down
df['Support_Tickets'] = (df['Support_Tickets'] / 4).astype(int)
df['Late_Payment_Count'] = (df['Late_Payment_Count'] / 3).astype(int)

# keeping satisfaction and NPS within their normal expected ranges
df['Customer_Satisfaction'] = df['Customer_Satisfaction'].clip(1, 10)
df['NPS_Score'] = df['NPS_Score'].clip(-100, 100)

print("Done! Support tickets and late payment metrics scaled successfully.")
print(f"Verified Ticket Range  : {df['Support_Tickets'].min()} to {df['Support_Tickets'].max()} tickets per user")
print(f"Verified Payment Range : {df['Late_Payment_Count'].min()} to {df['Late_Payment_Count'].max()} late flags")


Standardizing behavioral metrics and scaling complaint columns...
Done! Support tickets and late payment metrics scaled successfully.
Verified Ticket Range  : 0 to 6 tickets per user
Verified Payment Range : 0 to 5 late flags


In [19]:
# Removing the 'Churn' variable from the formula so the model doesn't cheat.
# Calculating a pure behavioral friction index out of 100.

print("Calculating account risk indices and customer lifetime values...")

# RISK SCORE
max_allowed_bill = df['Monthly_Bill'].max() if df['Monthly_Bill'].max() > 0 else 1
bill_debt_ratio = (df['Outstanding_Balance'] / max_allowed_bill) * 10

# Risk score is built from late payments, debt relative to bill, and support tickets
df['Risk_Score'] = (
    (df['Late_Payment_Count'] * 30) + 
    (bill_debt_ratio * 3.5) + 
    (df['Support_Tickets'] * 5)
)

# Cap the results cleanly between 0 and 100, converting to integers
df['Risk_Score'] = df['Risk_Score'].clip(0, 100).astype(int)


# 2. EXPECTED REVENUE ESTIMATION (CLV)
past_revenue_collected = df['Monthly_Bill'] * df['Tenure_Months']
projected_lifespan_months = np.where(df['Risk_Score'] > 50, 3, 24)
df['Estimated_CLV'] = (past_revenue_collected + (df['Monthly_Bill'] * projected_lifespan_months * 0.35)).round(2)

print("Finished adjusting financial metrics.")
print(f"True Baseline Risk Score Average: {df['Risk_Score'].mean():.1f}")
print(f"True Baseline Average Account CLV: ₦{df['Estimated_CLV'].mean():,.2f}")


Calculating account risk indices and customer lifetime values...
Finished adjusting financial metrics.
True Baseline Risk Score Average: 70.3
True Baseline Average Account CLV: ₦426,023.42


In [20]:
# Cleaning remaining categorical columns
# Employment status
df['Employment_Status'] = df['Employment_Status'].str.strip().str.title()
df['Employment_Status'] = df['Employment_Status'].fillna('Unknown')

# Internet type
df['Internet_Type'] = df['Internet_Type'].str.strip().str.title()
df['Internet_Type'] = df['Internet_Type'].fillna('Unknown')

# Contract type
df['Contract_Type'] = df['Contract_Type'].str.strip().str.title()
df['Contract_Type'] = df['Contract_Type'].fillna('Unknown')

# Payment method
df['Payment_Method'] = df['Payment_Method'].str.strip().str.title()
df['Payment_Method'] = df['Payment_Method'].fillna('Unknown')

# Payment status
df['Payment_Status'] = df['Payment_Status'].str.strip().str.title()
df['Payment_Status'] = df['Payment_Status'].fillna('Unknown')

# Complaint category
df['Complaint_Category'] = df['Complaint_Category'].str.strip().str.title()
df['Complaint_Category'] = df['Complaint_Category'].fillna('Unknown')

# Customer segment
df['Customer_Segment'] = df['Customer_Segment'].str.strip().str.title()
df['Customer_Segment'] = df['Customer_Segment'].fillna('Unknown')

# Churn
df['Churn'] = df['Churn'].str.strip().str.title()
df['Churn'] = df['Churn'].fillna('No')

print("All categorical columns cleaned.")

All categorical columns cleaned.


In [21]:
#Standardizing binary fields to clear out loose text case variants
def format_boolean_flags(value, default='No'):
    if pd.isna(value): 
        return default
    val_clean = str(value).strip().lower()
    return 'Yes' if val_clean in ['yes', 'y', '1', 'true'] else 'No'

for col in ['Roaming_Usage', 'Promotion_Clicked', 'Discount_Applied']:
    df[col] = df[col].apply(format_boolean_flags)

# 3. Cleaning up the final Monthly Data Usage column to ensure it is numerical
df['Monthly_Data_Usage_GB'] = pd.to_numeric(df['Monthly_Data_Usage_GB'], errors='coerce').fillna(0.0)

# Formating the datetime object to a clean string format for text export
df['Customer_Since'] = pd.to_datetime(df['Customer_Since']).dt.strftime('%Y-%m-%d')

# Drop any unused temporary tracking columns
if 'Customer_Since_Raw' in df.columns: 
    df.drop(columns=['Customer_Since_Raw'], inplace=True)


print("           SUMMARY")
print(f"Cleaned Database Shape  : {df.shape[0]:,} Rows | {df.shape[1]} Columns")
print(f"Remaining Missing Fields: {df.isnull().sum().sum()} null values")
print(f"Verified Average CLV    : ₦{df['Estimated_CLV'].mean():,.2f}")
print(f"Verified Churn Risk Index: {df['Risk_Score'].mean():.1f} Out of 100")

import os

# AUTOMATIC DIRECTORY CHECK 

destination_path = r'data/ADMIN/OneDrive/Projects/Telecom_Project/data/TelecomChurn_Nigeria_Clean.csv'
parent_folder = os.path.dirname(destination_path)

# If the folder path doesn't exist, build it dynamically on your hard drive
if not os.path.exists(parent_folder):
    os.makedirs(parent_folder)
    print(f"Created missing folder directory path: {parent_folder}")

# Saving overwritng the file completely
df.to_csv(destination_path, index=False)
print("File successfully saved")


           SUMMARY
Cleaned Database Shape  : 10,000 Rows | 39 Columns
Remaining Missing Fields: 0 null values
Verified Average CLV    : ₦426,023.42
Verified Churn Risk Index: 70.3 Out of 100
File successfully saved


## Model Development
Split the data, train the logistic regression model, and generate predictions using the cleaned dataset.

In [22]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

predictive_features = [
    'Age', 'Monthly_Income', 'Monthly_Data_Usage_GB', 'Monthly_Bill', 
    'Risk_Score', 'Customer_Satisfaction', 'NPS_Score', 
    'Mobile_App_Login_Last30Days', 'Tenure_Months'
]

X = df[predictive_features]
y = df['Churn'].map({'Yes': 1, 'No': 0}) 

# Spliting into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaling features so the model isn't biased toward larger-magnitude columns
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ADDING CLASS_WEIGHT
churn_classifier = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
churn_classifier.fit(X_train_scaled, y_train)

# Test the model's predictive ability on hidden rows
y_pred = churn_classifier.predict(X_test_scaled)

# Printing validation results
model_accuracy = accuracy_score(y_test, y_pred)

print("/n MODEL PERFORMANCE RESULTS")
print(f"Overall Model Accuracy: {model_accuracy * 100:.2f}%")
print("/nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("/n Classification Summary:")
print(classification_report(y_test, y_pred))



 MODEL PERFORMANCE RESULTS
Overall Model Accuracy: 50.40%

Confusion Matrix:
[[681 814]
 [178 327]]

 Classification Summary:
              precision    recall  f1-score   support

           0       0.79      0.46      0.58      1495
           1       0.29      0.65      0.40       505

    accuracy                           0.50      2000
   macro avg       0.54      0.55      0.49      2000
weighted avg       0.66      0.50      0.53      2000



## Model Evaluation
Evaluate the trained model using standard classification metrics to assess predictive performance.

In [23]:

# Checking the true weights of the non overlapping matrix.

model_weights = churn_classifier.coef_[0]  # Isolate the flat array out of the nested matrix

feature_comparison = pd.DataFrame({
    'Metric_Column': predictive_features,
    'Importance_Weight': model_weights
})

feature_comparison = feature_comparison.sort_values(by='Importance_Weight', ascending=False)

for index, row in feature_comparison.iterrows():
    if row['Importance_Weight'] > 0:
        print(f"{row['Metric_Column']}: {row['Importance_Weight']:.4f} (Increases Churn Risk)")
    else:
        print(f"{row['Metric_Column']}: {row['Importance_Weight']:.4f} (Reduces Churn Risk)")


Risk_Score: 0.1814 (Increases Churn Risk)
Monthly_Data_Usage_GB: 0.0068 (Increases Churn Risk)
Age: -0.0071 (Reduces Churn Risk)
Monthly_Income: -0.0101 (Reduces Churn Risk)
Mobile_App_Login_Last30Days: -0.0234 (Reduces Churn Risk)
Tenure_Months: -0.0253 (Reduces Churn Risk)
Monthly_Bill: -0.0372 (Reduces Churn Risk)
NPS_Score: -0.0398 (Reduces Churn Risk)
Customer_Satisfaction: -0.0790 (Reduces Churn Risk)


In [25]:
print(f"Overall Model Accuracy: {model_accuracy * 100:.2f}%")

Overall Model Accuracy: 50.40%
